<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# Python for Algorithmic Trading

**Chapter 04 &mdash; Vectorized Backtesting**

## Making Use of Vectorization

### Vectorization with NumPy 

In [ ]:
!git clone https://github.com/tpq-classes/python_for_algo_trading_core.git
import sys
sys.path.append('python_for_algo_trading_core')


In [ ]:
v = [1, 2, 3, 4, 5]

In [ ]:
sm = [2 * i for i in v]

In [ ]:
sm

In [ ]:
2 * v

In [ ]:
import numpy as np

In [ ]:
a = np.array(v)

In [ ]:
a

In [ ]:
type(a)

In [ ]:
2 * a

In [ ]:
0.5 * a + 2

In [ ]:
a = np.arange(12).reshape((4, 3))

In [ ]:
a

In [ ]:
2 * a

In [ ]:
a ** 2

In [ ]:
2 ** a

In [ ]:
a ** a

In [ ]:
a.mean()

In [ ]:
np.mean(a)

In [ ]:
a.mean(axis=0)

In [ ]:
np.mean(a, axis=1)

### Vectorization with pandas

In [ ]:
a = np.arange(15).reshape(5, 3)

In [ ]:
a

In [ ]:
import pandas as pd

In [ ]:
columns = list('abc')

In [ ]:
columns

In [ ]:
index = pd.date_range('2021-7-1', periods=5, freq='B')

In [ ]:
index

In [ ]:
df = pd.DataFrame(a, columns=columns, index=index)

In [ ]:
df

In [ ]:
2 * df

In [ ]:
df.sum()

In [ ]:
np.mean(df, axis=0)

In [ ]:
df['a'] + df['c']

In [ ]:
0.5 * df.a + 2 * df.b - df.c

In [ ]:
df['a'] > 5

In [ ]:
df[df['a'] > 5]

In [ ]:
df['c'] > df['b']

In [ ]:
0.15 * df.a + df.b > df.c

## Strategies based on Simple Moving Averages

### Getting into the Basics 

In [ ]:
raw = pd.read_csv('http://hilpisch.com/pyalgo_eikon_eod_data.csv',
                   index_col=0, parse_dates=True).dropna()

In [ ]:
raw.info()

In [ ]:
data = pd.DataFrame(raw['EUR='])

In [ ]:
data.rename(columns={'EUR=': 'price'}, inplace=True)

In [ ]:
data.info()

In [ ]:
data['SMA1'] = data['price'].rolling(42).mean()

In [ ]:
data['SMA2'] = data['price'].rolling(252).mean()

In [ ]:
data.tail()

In [ ]:
%matplotlib inline
from pylab import mpl, plt
plt.style.use('seaborn-v0_8')
mpl.rcParams['font.family'] = 'serif'

In [ ]:
data.plot(title='EUR= stock price | 42 & 252 days SMAs',
          figsize=(10, 6));

In [ ]:
data['position'] = np.where(data['SMA1'] > data['SMA2'],
                            1, -1)

In [ ]:
data.head()

In [ ]:
data.dropna(inplace=True)

In [ ]:
data.head()

In [ ]:
data['position'].plot(ylim=[-1.1, 1.1],
                      title='Market Positioning',
                      figsize=(10, 6));

In [ ]:
ax = data.plot(title='EUR= stock price | 42 & 252 days SMAs',
          figsize=(10, 6), secondary_y='position');

In [ ]:
data['returns'] = np.log(data['price'] / data['price'].shift(1))

In [ ]:
data['returns'].hist(bins=35, figsize=(10, 6));

In [ ]:
data['strategy'] = data['position'].shift(1) * data['returns']

In [ ]:
data.head()

In [ ]:
data[['returns', 'strategy']].sum()

In [ ]:
data[['returns', 'strategy']].sum().apply(np.exp)

In [ ]:
data[['returns', 'strategy']].dropna().cumsum(
            ).apply(np.exp).plot(figsize=(10, 6));

In [ ]:
data[['returns', 'strategy']].mean() * 252

In [ ]:
data[['returns', 'strategy']].std() * 252 ** 0.5

In [ ]:
data['cumret'] = data['strategy'].cumsum().apply(np.exp)

In [ ]:
data['cummax'] = data['cumret'].cummax()

In [ ]:
data[['cumret', 'cummax']].dropna().plot(figsize=(10, 6));

In [ ]:
drawdown = data['cummax'] - data['cumret']

In [ ]:
drawdown.max()

In [ ]:
temp = drawdown[drawdown == 0]

In [ ]:
periods = (temp.index[1:].to_pydatetime() -
           temp.index[:-1].to_pydatetime())

In [ ]:
periods[12:20]

In [ ]:
periods.max()

### Generalizing the Approach

In [ ]:
import SMAVectorBacktester as SMA

In [ ]:
smabt = SMA.SMAVectorBacktester('EUR=', 42, 252,
                                '2010-1-1', '2019-12-31') 

In [ ]:
smabt.run_strategy()

In [ ]:
smabt.plot_results()

In [ ]:
%%time
smabt.optimize_parameters((30, 50, 2),
                          (200, 300, 2))

In [ ]:
smabt.plot_results()

## Strategies based on Momentum

### Getting into the Basics

In [ ]:
data = pd.DataFrame(raw['XAU='])

In [ ]:
data.rename(columns={'XAU=': 'price'}, inplace=True)

In [ ]:
data['returns'] = np.log(data['price'] / data['price'].shift(1))

In [ ]:
data['position'] = np.sign(data['returns'])

In [ ]:
data['strategy'] = data['position'].shift(1) * data['returns']

In [ ]:
data[['returns', 'strategy']].dropna().cumsum(
            ).apply(np.exp).plot(figsize=(10, 6));

In [ ]:
data['position'] = np.sign(data['returns'].rolling(3).mean())

In [ ]:
data.head()

In [ ]:
data['strategy'] = data['position'].shift(1) * data['returns']

In [ ]:
data[['returns', 'strategy']].dropna().cumsum(
        ).apply(np.exp).plot(figsize=(10, 6));

In [ ]:
fn = 'AAPL_1min_05052020.csv'
fn = 'SPX_1min_05052020.csv'

In [ ]:
data = pd.read_csv(fn, index_col=0, parse_dates=True)

In [ ]:
data.info()

In [ ]:
data['returns'] = np.log(data['CLOSE'] /
                         data['CLOSE'].shift(1))

In [ ]:
to_plot = ['returns']

In [ ]:
for m in [1, 3, 5, 7, 9]:
    data['position_%d' % m] = np.sign(data['returns'].rolling(m).mean())
    data['strategy_%d' % m] = (data['position_%d' % m].shift(1) *
                               data['returns'])
    to_plot.append('strategy_%d' % m)

In [ ]:
data[to_plot].dropna().cumsum().apply(np.exp).plot(
    title='SPX intraday 05. May 2020',
    figsize=(10, 6), style=['-', '--', '--', '--', '--', '--']);

### Generalizing the Approach

In [ ]:
import MomVectorBacktester as Mom

In [ ]:
mombt = Mom.MomVectorBacktester('XAU=', '2010-1-1',
                                '2019-12-31', 10000, 0.0)

In [ ]:
mombt.run_strategy(momentum=3)

In [ ]:
mombt.plot_results()

In [ ]:
mombt = Mom.MomVectorBacktester('XAU=', '2010-1-1',
                                '2019-12-31', 10000, 0.001)

In [ ]:
mombt.run_strategy(momentum=3)

In [ ]:
mombt.plot_results()

## Strategies based on Mean-Reversion

### Getting into the Basics

In [ ]:
data = pd.DataFrame(raw['GDX'])

In [ ]:
data.rename(columns={'GDX': 'price'}, inplace=True)

In [ ]:
data['returns'] = np.log(data['price'] /
                         data['price'].shift(1))

In [ ]:
SMA = 25

In [ ]:
data['SMA'] = data['price'].rolling(SMA).mean()

In [ ]:
threshold = 3.5

In [ ]:
data['distance'] = data['price'] - data['SMA']

In [ ]:
data['distance'].dropna().plot(figsize=(10, 6), legend=True)
plt.axhline(threshold, color='r', ls='--')
plt.axhline(-threshold, color='r', ls='--')
plt.axhline(0, color='r');

In [ ]:
data['position'] = np.where(data['distance'] > threshold,
                            -1, np.nan)

In [ ]:
data['position'] = np.where(data['distance'] < -threshold,
                            1, data['position'])

In [ ]:
data['position'] = np.where(data['distance'] *
            data['distance'].shift(1) < 0, 0, data['position'])

In [ ]:
data['position'] = data['position'].ffill().fillna(0)

In [ ]:
data['position'].iloc[SMA:].plot(ylim=[-1.1, 1.1],
                               figsize=(10, 6));

In [ ]:
data['strategy'] = data['position'].shift(1) * data['returns']

In [ ]:
data[['returns', 'strategy']].dropna().cumsum(
        ).apply(np.exp).plot(figsize=(10, 6));

### Generalizing the Approach 

In [ ]:
import MRVectorBacktester as MR

In [ ]:
mrbt = MR.MRVectorBacktester('GLD', '2010-1-1', '2019-12-31',
                             10000, 0.001)

In [ ]:
mrbt.run_strategy(SMA=43, threshold=7.5)

In [ ]:
mrbt.plot_results()

### Brute Force Optimization

In [ ]:
%%time
r = list()
for SMA in range(5, 65):
    for threshold in np.arange(2, 15, 0.5):
        res = mrbt.run_strategy(SMA=SMA, threshold=threshold)
        r.append((res, SMA, threshold))
r.sort()

In [ ]:
r[::-1][:5]

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

<a href="http://tpq.io" target="_blank">http://tpq.io</a> | <a href="http://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:training@tpq.io">training@tpq.io</a>